# SBIR Review Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSV-AI/agent-playground/blob/dev/notebooks/sbir_review.ipynb)

## Description
This notebook replicates the functionality of the `src/agent_playground/sbir_review` agent, demonstrating web scraping and data extraction capabilities using PydanticAI and Playwright.

## Setup

Before running this notebook, ensure you have the required dependencies installed and your OpenRouter API key configured.


### Install required Python dependencies

In [ ]:
!pip install pydantic-ai pydantic python-dotenv

### Install npx playwright dependencies

This cell installs Playwright and its necessary browser drivers.

In [ ]:
!npx playwright install

### Import necessary libraries

In [ ]:
import os
import asyncio
from pydantic import BaseModel, Field
from pydantic_ai import Agent, AgentRunResult
from pydantic_ai.mcp import MCPServerStdio, load_mcp_servers
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
import logfire
import json
from typing import List, Any
from datetime import datetime
from pathlib import Path
from google.colab import userdata


## Define Data Models

These models are identical to those found in `src/agent_playground/sbir_review/models.py`.

In [ ]:
class Topic(BaseModel):
    """Structured data about a single product."""
    title: str = Field(description="The title of the topic.")
    open_date: datetime = Field(description="The date when the topic was opened.")
    close_date: datetime = Field(description="The date when the topic was closed.")
    component: str = Field(description="The component associated with the topic. This should be an acronym in all capital letters.")
    technology_areas: List[str] = Field(description="A list of technology areas related to the topic.")
    modernization_priorities: List[str] = Field(description="A list of modernization priorities for the topic.")
    keywords: List[str] = Field(description="A list of keywords associated with the topic.")
    objective: str = Field(description="The main objective of the topic.")
    phases: str = Field(description="The phases involved in the topic. This should be a combination of Phase 1, Phase 2, and Phase 3 as applicable.")
    references: List[str] = Field(description="A list of references or links related to the topic.")

class ScrapeResult(BaseModel):
    """The collection of all extracted product information."""
    topics: List[Topic]
    error_message: str = Field(description="An error message if the scraping failed, otherwise empty.")


## Configure LLM and MCP Server

Configure PydanticAI to use OpenRouter with an OpenAI-compatible model and load the MCP server configuration for Playwright.

In [ ]:
# --- Configuration ---
# Since OpenRouter has a unified API, we can use the OpenAIChatModel with a custom provider.
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_KEY = userdata.get('OPENROUTER_API_KEY') # For Colab secrets

# 1. Configure the LLM for OpenRouter
_model = OpenAIChatModel(
    OPENROUTER_MODEL,
    provider=OpenRouterProvider(
        api_key=OPENROUTER_KEY
    ),
)

# MCP Server Configuration (from src/agent_playground/sbir_review/mcp_config.json)
# This configuration assumes npx and @playwright/mcp are installed.
mcp_config_json = {
  "mcpServers": {
    "playwright": {
      "command": "npx",
      "args": [
       "@playwright/mcp",
       "--headless",
       "--browser",
       "chromium"
      ]
    }
  }
}

# The mcpServers require a file path, so we'll write this config to a temporary file
mcp_config_path = Path("mcp_config.json")
mcp_config_path.write_text(json.dumps(mcp_config_json))

_toolsets = load_mcp_servers(str(mcp_config_path))


## System Prompt

The system prompt guides the agent's behavior. This is directly from `src/agent_playground/sbir_review/system_prompt.md`.

In [ ]:
system_prompt_text = """
# Expert Web Scraping and Data Extraction Agent

You are a specialized Web Scraping Agent. Your sole objective is to autonomously fulfill the user's data request using the provided BrowserAutomation toolset.

# Core Directives & Procedure

1. Strict Tool Use: You MUST use the BrowserAutomation tools for all navigation, interaction, and data extraction tasks. Do not attempt to guess or hallucinate content.
2. Navigation: Start by calling goto_page(url) with the target URL provided by the user.
3. Dynamic Interaction: If required, interact with dynamic page elements (e.g., clicking buttons, filling forms) using appropriate tools before attempting extraction.
4. Extraction: Use the most precise tool available (e.g., extract_json_from_element, extract_list_of_topics) to gather the requested data.

# Exit Strategy and Error Handling

Your exit strategy must be based on the outcome of your operations:

1. Success: If you successfully gather all the requested data, populate the ScrapeResult JSON schema with the extracted information and return it immediately.
2. Failure/Error: If you encounter any of the following issues, you MUST ABORT the task and return the designated failure schema:
  - The goto_page tool reports a loading error (e.g., 404, connection timeout, infinite redirect).
  - The extract tool returns no relevant data despite successfully loading the page.
  - You exhaust the maximum number of tool call retries.
3. Failure Schema: Upon failure, return the ScrapeResult object with the topics or primary data list set to empty ([]) and include a brief explanation of the problem in a relevant field (e.g., error_message).

# Final Output Requirement

Your final action MUST be to return the result object defined by the ScrapeResult JSON schema. Do not include any conversational wrapper text in the final output.
"""


## Initialize the Agent

Create the Agent instance with the configured LLM, output type, toolsets, and system prompt.

In [ ]:
agent = Agent(
    model=_model,
    output_type=ScrapeResult,
    toolsets=_toolsets,
    system_prompt=system_prompt_text,
)


## User Prompt

This is the task the agent will perform, pulled directly from `src/agent_playground/sbir_review/user_prompt.md`.

In [ ]:
user_prompt_text = """
Go to the dynamic topics page at 'https://www.dodsbirsttr.mil/topics-app/'.
Scroll through the list of topics to load all available entries.
Extract the list of all topics available along with their title, open_date, close_date, and component.
For each topic, select the row to expand it. Wait up to 5 seconds for the content in each expanded section to load.
Extract the technology_areas, modernization_priorities, keywords, objective, phases, and references.
Combine all text from Phase I, Phase II, and Phase III into the phases field.
"""


## Run the Agent

Execute the agent with the user prompt.

In [ ]:
# configure logfire (optional)
LOGFIRE_TOKEN = os.environ.get('LOGFIRE_TOKEN')
if LOGFIRE_TOKEN:
  logfire.configure(token=LOGFIRE_TOKEN)
  logfire.instrument_pydantic_ai()

print(f"Running Agent Task: {user_prompt_text}\n")

# Run the agent, specifying the desired output structure
result = await agent.run(
    user_prompt_text,
 )

print("Task Complete! Extracted Structured Data:")
for topic in result.output.topics:
    print(topic)

# Clean up the temporary mcp_config.json file
mcp_config_path.unlink()
